In [1]:
from google.colab import files

uploaded = files.upload()
uploaded = files.upload()

Saving testing_disease.csv to testing_disease.csv


Saving training_disease.csv to training_disease.csv


In [8]:
import pandas as pd
#storing is train and test variable
train = pd.read_csv("/content/training_disease.csv")
test = pd.read_csv("/content/testing_disease.csv")

#print(train.head())
#print(test.head())

In [9]:
train.isnull().sum()

,0
itching,0
skin_rash,0
nodal_skin_eruptions,0
continuous_sneezing,0
shivering,0
...,...
inflammatory_nails,0
blister,0
red_sore_around_nose,0
yellow_crust_ooze,0


In [10]:
test.isnull().sum()

,0
itching,0
skin_rash,0
nodal_skin_eruptions,0
continuous_sneezing,0
shivering,0
...,...
inflammatory_nails,0
blister,0
red_sore_around_nose,0
yellow_crust_ooze,0


In [11]:
print("test data duplicate:",test.duplicated().sum())
print("train data duplicate",train.duplicated().sum())

test data duplicate: 0
train data duplicate 4616


In [27]:
train.drop_duplicates(inplace=True)


In [28]:
print("test data duplicate:",test.duplicated().sum())
print("train data duplicate",train.duplicated().sum())

test data duplicate: 0
train data duplicate 0


In [29]:
print("test shape",test.shape)
print("train shape",train.shape)

test shape (41, 133)
train shape (304, 133)


In [30]:
train.columns

Index(['itching', 'skin_rash', 'nodal_skin_eruptions', 'continuous_sneezing',
       'shivering', 'chills', 'joint_pain', 'stomach_pain', 'acidity',
       'ulcers_on_tongue',
       ...
       'blackheads', 'scurring', 'skin_peeling', 'silver_like_dusting',
       'small_dents_in_nails', 'inflammatory_nails', 'blister',
       'red_sore_around_nose', 'yellow_crust_ooze', 'prognosis'],
      dtype='object', length=133)

In [31]:
X_train = train.drop('prognosis', axis=1)
y_train = train['prognosis']

In [32]:
X_test = test.drop('prognosis', axis=1)
y_test = test['prognosis']

In [33]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier

In [34]:
#creating the models
# Decision Tree
dt = DecisionTreeClassifier(random_state=42)

# Random Forest
rf = RandomForestClassifier(n_estimators=100, random_state=42)

# Naive Bayes
nb = GaussianNB()

# K-Nearest Neighbors
knn = KNeighborsClassifier(n_neighbors=5)

In [35]:
#fitting the models
dt.fit(X_train, y_train)


DecisionTreeClassifier(random_state=42)

In [36]:
knn.fit(X_train, y_train)

KNeighborsClassifier()

In [37]:
nb.fit(X_train, y_train)

GaussianNB()

In [38]:
rf.fit(X_train, y_train)

RandomForestClassifier(random_state=42)

In [39]:
#storing the predictions
dt_pred = dt.predict(X_test)

rf_pred = rf.predict(X_test)

nb_pred = nb.predict(X_test)

knn_pred = knn.predict(X_test)

In [40]:
from sklearn.metrics import accuracy_score

print("Decision Tree Accuracy:", accuracy_score(y_test, dt_pred))
print("Random Forest Accuracy:", accuracy_score(y_test, rf_pred))
print("Naive Bayes Accuracy:", accuracy_score(y_test, nb_pred))
print("KNN Accuracy:", accuracy_score(y_test, knn_pred))

Decision Tree Accuracy: 1.0
Random Forest Accuracy: 1.0
Naive Bayes Accuracy: 1.0
KNN Accuracy: 1.0


In [42]:
from sklearn.metrics import accuracy_score

dt_accuracy = f"{accuracy_score(y_test, dt_pred)*100:.2f}%"
rf_accuracy = f"{accuracy_score(y_test, rf_pred)*100:.2f}%"
nb_accuracy = f"{accuracy_score(y_test, nb_pred)*100:.2f}%"
knn_accuracy = f"{accuracy_score(y_test, knn_pred)*100:.2f}%"

In [43]:
#we are using gradio user interface
import gradio as gr

# Symptom List


symptoms = [
    "Select Here",
    "itching",
    "skin_rash",
    "continuous_sneezing",
    "shivering",
    "joint_pain",
    "stomach_pain",
    "acidity",
    "vomiting",
    "fatigue",
    "weight_loss",
    "cough",
    "high_fever",
    "headache",
    "back_pain",
    "constipation",
    "diarrhoea",
    "nausea",
    "chest_pain",
    "neck_pain",
    "dizziness",
    "loss_of_balance",
    "loss_of_smell"
]


# Prediction Function


import numpy as np

def predict(name, s1, s2, s3, s4, s5):

    if name.strip() == "":
        return (
            "Enter Patient Name","","",
            "", "", "",

            "", "", "",

            "", "", ""
        )

    selected = [s for s in [s1, s2, s3, s4, s5] if s != "Select Here"]

    if len(selected) < 3:
        return (
            "Select minimum 3 symptoms","","",
            "", "", "",

            "", "", "",

            "", "", ""
        )

    # Create input vector
    input_vector = np.zeros(len(X_train.columns))

    for symptom in selected:
        if symptom in X_train.columns:
            input_vector[X_train.columns.get_loc(symptom)] = 1

    input_vector = input_vector.reshape(1, -1)

    # Predictions
    dt_prediction = dt.predict(input_vector)[0]
    rf_prediction = rf.predict(input_vector)[0]
    nb_prediction = nb.predict(input_vector)[0]
    knn_prediction = knn.predict(input_vector)[0]

    # Confidence (%)
    dt_conf = np.max(dt.predict_proba(input_vector)) * 100
    rf_conf = np.max(rf.predict_proba(input_vector)) * 100
    nb_conf = np.max(nb.predict_proba(input_vector)) * 100
    knn_conf = np.max(knn.predict_proba(input_vector)) * 100

    return (

        dt_prediction,
      dt_accuracy,
      f"{dt_conf:.2f}%",

      rf_prediction,
      rf_accuracy,
      f"{rf_conf:.2f}%",

      nb_prediction,
      nb_accuracy,
      f"{nb_conf:.2f}%",

      knn_prediction,
      knn_accuracy,
      f"{knn_conf:.2f}%"

    )


# ==========================
# Custom CSS
# ==========================

css = """

h1{
text-align:center;
color:#ff1493;
font-size:40px;
}

.subtitle{
text-align:center;
font-size:22px;
color:purple;
font-weight:bold;
margin-bottom:20px;
}

.result{
background:#d8ffd8;
padding:12px;
border-radius:8px;
font-size:18px;
font-weight:bold;
}

"""

# ==========================
# UI
# ==========================

with gr.Blocks(css=css,title="Disease Prediction") as demo:

    gr.Markdown(
        "# DISEASE PREDICTION USING MACHINE LEARNING"
    )

    gr.Markdown(
        "<div class='subtitle'>Enter minimum three symptoms to get prediction</div>"
    )

    with gr.Row():

        with gr.Column(scale=1):

            name = gr.Textbox(
                label="Patient Name",
                placeholder="Enter Patient Name"
            )

            symptom1 = gr.Dropdown(symptoms,label="Symptom 1")
            symptom2 = gr.Dropdown(symptoms,label="Symptom 2")
            symptom3 = gr.Dropdown(symptoms,label="Symptom 3")
            symptom4 = gr.Dropdown(symptoms,label="Symptom 4")
            symptom5 = gr.Dropdown(symptoms,label="Symptom 5")

            with gr.Row():

                predict_btn = gr.Button(
                    "Predict Disease",
                    variant="primary"
                )

                clear_btn = gr.ClearButton()

        with gr.Column(scale=1):

            gr.Markdown("## 🌳 Decision Tree")

            dt_result = gr.Textbox(label="Prediction")

            dt_acc = gr.Textbox(label="Accuracy")

            dt_conf = gr.Textbox(label="Confidence")

            gr.Markdown("---")

            gr.Markdown("## 🌲 Random Forest")

            rf_result = gr.Textbox(label="Prediction")

            rf_acc = gr.Textbox(label="Accuracy")
            rf_conf = gr.Textbox(label="Confidence")

        with gr.Column(scale=1):

            gr.Markdown("## 📊 Naive Bayes")

            nb_result = gr.Textbox(label="Prediction")

            nb_acc = gr.Textbox(label="Accuracy")
            nb_conf = gr.Textbox(label="Confidence")

            gr.Markdown("---")

            gr.Markdown("## 👥 KNN")

            knn_result = gr.Textbox(label="Prediction")

            knn_acc = gr.Textbox(label="Accuracy")
            knn_conf = gr.Textbox(label="Confidence")

    predict_btn.click(

        predict,

        inputs=[
            name,
            symptom1,
            symptom2,
            symptom3,
            symptom4,
            symptom5,
        ],

        outputs=[
            dt_result,
            dt_acc,
            dt_conf,
            rf_result,
            rf_acc,
            rf_conf,
            nb_result,
            nb_acc,
            nb_conf,
            knn_result,
            knn_acc,
            knn_conf
        ],

    )

demo.launch()

/tmp/ipykernel_1618/3566128054.py:139: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: css. Please pass these parameters to launch() instead.
  with gr.Blocks(css=css,title="Disease Prediction") as demo:


It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://a3d5eb7b6a0f9371d7.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
